# Desafio Técnico - Gato Mestre (Ciência de Dados)
## Notebook 05: Otimização de Hiperparâmetros, Explicabilidade com SHAP e Entrega de Previsões

### 🎯 Objetivo do Notebook
1. **Otimização Bayesiana (*Hyperparameter Tuning*):** Utilizar o framework **Optuna** com amostrador TPE (*Tree-structured Parzen Estimator*) para refinar os hiperparâmetros do modelo campeão (**LightGBM**), minimizando o erro na safra de Validação (2024).
2. **Explicabilidade Global e Local (*SHAP Values*):** Aplicar o algoritmo *TreeSHAP* para auditar a contribuição marginal de cada feature na predição de pontuação dos atletas.
3. **Geração do Entregável Oficial (`previsoes.json`):** Exportar as previsões da safra de teste OOS (2025) no contrato estrito exigido pelo produto Gato Mestre / Cartola FC.

### 1. Importação das Bibliotecas, Semente de Reprodutibilidade e Carga dos Dados

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display
import sys
import warnings
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import joblib

# Otimização Bayesiana & Explicabilidade
import optuna
import shap
from lightgbm import LGBMRegressor

# Métricas
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Tabelas interativas
import itables
from itables import init_notebook_mode, show

# Configurações estéticas e logs
warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)
plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")

init_notebook_mode(all_interactive=True)
itables.options.maxBytes = 0
itables.options.classes = ["display", "nowrap"]
itables.options.lengthMenu = [10, 25, 50]

# Semente Global para 100% de Reprodutibilidade
GLOBAL_SEED = 42
np.random.seed(GLOBAL_SEED)

# Caminhos do projeto
PROJECT_ROOT = Path("..").resolve() if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

DATA_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Carga da matriz consolidada de features (115.613 linhas)
df_features = pd.read_parquet(DATA_PROCESSED_DIR / "base_features_gm.parquet")
print(f"Matriz de Features carregada: {df_features.shape[0]:,} linhas x {df_features.shape[1]} colunas")
print(f"Total de nulos: {df_features.isna().sum().sum()} (100% íntegra)")
print(f"Semente Global Ativa: {GLOBAL_SEED}")

### 2. Estratégia de Particionamento Temporal Estrito

Mantemos a divisão cronológica sem vazamento de dados (*data leakage*):
* 🏛️ **Treino ($2022 + 2023$):** $59.213$ registros para ajuste das árvores.
* 🔍 **Validação ($2024$):** $28.568$ registros avaliados na função objetivo do Optuna.
* 🚀 **Teste Out-of-Sample ($2025$):** $27.832$ registros para avaliação cega final e geração do `previsoes.json`.

In [ ]:
cat_cols = ["posicao_id", "status_pre", "status_inicial", "clube_id", "opponent"]
bin_cols = ["home_dummy", "is_inicio_temporada", "participou_lag1"]
num_cols = [
    "rodada_id", "preco_num", "variacao_num", "media_num", "jogos_num",
    "clube_media_pontos_conquistados", "opponent_media_pontos_cedidos",
    "progresso_campeonato", "taxa_participacao_3j", "minutos_medios_3j",
    "pontos_lag1", "media_pontos_3j", "desvio_pontos_3j", "media_scouts_volume_3j",
    "momentum_preco_3j", "roi_recente_3j", "estabilidade_11_titular_clube",
    "fator_alavancagem_confronto", "potencial_esperado_atleta",
    "indice_favoritismo_mando", "volume_esperado_partida",
    "diff_forca_confronto", "score_risco_rotacao"
]
feature_cols = cat_cols + num_cols + bin_cols

# Codificação Categórica Ordinal para Árvores
df_encoded = df_features.copy()
for col in cat_cols:
    df_encoded[col] = df_encoded[col].astype("category").cat.codes

train_mask = df_features["ano"].isin([2022, 2023])
val_mask = df_features["ano"] == 2024
test_mask = df_features["ano"] == 2025

X_train = df_encoded.loc[train_mask, feature_cols]
y_train = df_encoded.loc[train_mask, "pontos_num"].values

X_val = df_encoded.loc[val_mask, feature_cols]
y_val = df_encoded.loc[val_mask, "pontos_num"].values

X_test = df_encoded.loc[test_mask, feature_cols]
y_test = df_encoded.loc[test_mask, "pontos_num"].values

print(f"X_train: {X_train.shape} | y_train: {len(y_train):,}")
print(f"X_val:   {X_val.shape}   | y_val:   {len(y_val):,}")
print(f"X_test:  {X_test.shape}  | y_test:  {len(y_test):,}")

### 3. Otimização Bayesiana de Hiperparâmetros (*Optuna*)

Configuramos o estudo com **Tree-structured Parzen Estimator (TPE)** para explorar de forma inteligente o espaço contínuo e discreto de regularização, taxa de aprendizado e estrutura de divisão das árvores.

In [ ]:
def objective(trial):
    params = {
        "objective": "regression",
        "metric": "mae",
        "boosting_type": "gbdt",
        "random_state": GLOBAL_SEED,
        "verbose": -1,
        "n_estimators": trial.suggest_int("n_estimators", 100, 350, step=50),
        "learning_rate": trial.suggest_float("learning_rate", 0.02, 0.08, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 20, 63),
        "max_depth": trial.suggest_int("max_depth", 4, 9),
        "min_child_samples": trial.suggest_int("min_child_samples", 15, 60),
        "subsample": trial.suggest_float("subsample", 0.70, 0.95),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.70, 0.95),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 5.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 5.0, log=True),
    }
    
    model = LGBMRegressor(**params)
    model.fit(X_train, y_train)
    p_val = model.predict(X_val)
    return mean_absolute_error(y_val, p_val)

print("Iniciando Otimização Bayesiana com Optuna (40 Trials)...\n")
study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=GLOBAL_SEED))
study.optimize(objective, n_trials=40, timeout=180)

print("\n=== OTIMIZAÇÃO CONCLUÍDA COM SUCESSO! ===")
print(f"Melhor MAE na Validação (2024): {study.best_value:.4f} pts")
print("\nMelhores Hiperparâmetros Encontrados:")
for param_name, param_val in study.best_params.items():
    print(f"  • {param_name}: {param_val}")

# Visualização da Convergência dos Trials
valores_trials = [t.value for t in study.trials if t.value is not None]
plt.figure(figsize=(10, 4))
plt.plot(range(1, len(valores_trials) + 1), valores_trials, marker="o", color="#2b5c8f", alpha=0.6, label="Trial MAE")
plt.axhline(study.best_value, color="red", linestyle="--", label=f"Melhor MAE: {study.best_value:.4f}")
plt.title("Histórico de Otimização Bayesiana do LightGBM (Optuna)", fontsize=12, pad=12)
plt.xlabel("Número do Trial", fontsize=11)
plt.ylabel("MAE na Safra de Validação (2024)", fontsize=11)
plt.legend(fontsize=10)
plt.tight_layout()
plt.show()

### 4. Comparativo de Performance no Teste OOS (Safra 2025 - $N = 27.832$)

Avaliamos o modelo default vs. o modelo com os hiperparâmetros otimizados pelo Optuna na safra cega de 2025:

In [ ]:
# 1. Treina Modelo Default
lgbm_default = LGBMRegressor(n_estimators=150, max_depth=6, learning_rate=0.04, random_state=GLOBAL_SEED, verbose=-1)
lgbm_default.fit(X_train, y_train)
p_test_default = lgbm_default.predict(X_test)

# 2. Treina Modelo Tunado com os Melhores Parâmetros
lgbm_tuned = LGBMRegressor(**study.best_params, random_state=GLOBAL_SEED, verbose=-1)
lgbm_tuned.fit(X_train, y_train)
p_test_tuned = lgbm_tuned.predict(X_test)

# Função de Avaliação Completa com 6 Métricas
top20_mask_test = y_test >= np.quantile(y_test, 0.80)

def avaliar_modelo(y_true, y_pred, nome):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    r_s, _ = stats.spearmanr(y_true, y_pred)
    r_p, _ = stats.pearsonr(y_true, y_pred)
    mae_top20 = mean_absolute_error(y_true[top20_mask_test], y_pred[top20_mask_test])
    return {
        "Modelo": nome,
        "MAE (Geral)": mae,
        "RMSE": rmse,
        "R²": r2,
        "Spearman (rho)": r_s,
        "Pearson (r)": r_p,
        "MAE Top 20%": mae_top20
    }

df_comp_final = pd.DataFrame([
    avaliar_modelo(y_test, df_features.loc[test_mask, "media_num"].values, "0. Baseline (media_num)"),
    avaliar_modelo(y_test, p_test_default, "1. LightGBM (Default)"),
    avaliar_modelo(y_test, p_test_tuned, "2. LightGBM (Tunado Optuna)")
])

print("=== COMPARAÇÃO FINAL DE PERFORMANCE NA SAFRA DE TESTE OOS (2025) ===")
display(df_comp_final.round(4))

# Gráfico Comparativo de Redução de Erro
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

cores_comp = ["#e74c3c", "#3498db", "#27ae60"]
axes[0].bar(df_comp_final["Modelo"], df_comp_final["MAE (Geral)"], color=cores_comp, edgecolor="black", width=0.5)
axes[0].set_title("MAE Geral (Menor é Melhor)", fontsize=11)
axes[0].set_ylabel("MAE (Pontos)")
axes[0].tick_params(axis="x", rotation=15)

axes[1].bar(df_comp_final["Modelo"], df_comp_final["R²"], color=cores_comp, edgecolor="black", width=0.5)
axes[1].set_title("R² - Variância Explicada (Maior é Melhor)", fontsize=11)
axes[1].set_ylabel("R²")
axes[1].tick_params(axis="x", rotation=15)

axes[2].bar(df_comp_final["Modelo"], df_comp_final["Spearman (rho)"], color=cores_comp, edgecolor="black", width=0.5)
axes[2].set_title("Spearman rho (Ranqueamento)", fontsize=11)
axes[2].set_ylabel("Spearman rho")
axes[2].tick_params(axis="x", rotation=15)

plt.tight_layout()
plt.show()

### 5. Interpretabilidade e Explicabilidade com *SHAP Values* (*TreeSHAP*)

Para auditar a física interna do modelo e garantir que ele toma decisões alinhadas com as regras do futebol:
* Calculamos os valores de Shapley usando o algoritmo exato **TreeSHAP**.
* Analisamos a importância global e a direção do impacto de cada variável preditora.

In [ ]:
print("Calculando SHAP Values com TreeExplainer no conjunto de Teste (Safra 2025)...\n")

# Amostra representativa para cálculo ágil dos SHAP values
np.random.seed(GLOBAL_SEED)
sample_indices = np.random.choice(X_test.index, size=min(4000, len(X_test)), replace=False)
X_test_shap = X_test.loc[sample_indices]

explainer = shap.TreeExplainer(lgbm_tuned)
shap_values = explainer.shap_values(X_test_shap)

print(f"SHAP values calculados com sucesso para {len(X_test_shap):,} atletas do Teste!")

# 1. SHAP Summary Plot (Beeswarm: Impacto e Direção)
plt.figure(figsize=(12, 7))
plt.title("1. SHAP Summary Plot (Beeswarm): Impacto Marginal de Cada Feature", fontsize=13, pad=15)
shap.summary_plot(shap_values, X_test_shap, max_display=15, show=False)
plt.tight_layout()
plt.show()

# 2. SHAP Feature Importance (Bar Plot)
plt.figure(figsize=(10, 6))
plt.title("2. Ranking de Importância das Features (Média do Impacto Absoluto SHAP)", fontsize=13, pad=15)
shap.summary_plot(shap_values, X_test_shap, plot_type="bar", max_display=15, show=False)
plt.tight_layout()
plt.show()

### 6. Exportação do Modelo Tunado e Geração do `previsoes.json` Oficial

Exportamos o modelo campeão serializado (`models/modelo_lightgbm_tunado.joblib`) e construímos o arquivo `previsoes.json` estritamente no contrato exigido pelo desafio:

In [ ]:
# 1. Salva o Modelo Tunado Serializado (.joblib)
CAMINHO_MODELO_TUNADO = MODELS_DIR / "modelo_lightgbm_tunado.joblib"
joblib.dump(lgbm_tuned, CAMINHO_MODELO_TUNADO)
print(f"Modelo LightGBM Tunado salvo em: {CAMINHO_MODELO_TUNADO} ({CAMINHO_MODELO_TUNADO.stat().st_size / (1024*1024):.2f} MB)")

# 2. Salva a Matriz Consolidada de Predições da Safra 2025 em Parquet
df_pred_2025 = df_features[test_mask][[
    "ano", "rodada_id", "atleta_id", "apelido", "clube_id", "posicao_id", "posicao_nome",
    "status_pre", "entrou_em_campo", "preco_num", "home_dummy", "opponent", "pontos_num"
]].copy()
df_pred_2025["pred_lightgbm_default"] = p_test_default
df_pred_2025["pontos_predito"] = np.round(p_test_tuned, 2)

CAMINHO_PARQUET_2025 = DATA_PROCESSED_DIR / "predicoes_modelos_2025.parquet"
df_pred_2025.to_parquet(CAMINHO_PARQUET_2025, index=False)
print(f"Base Parquet 2025 salva em: {CAMINHO_PARQUET_2025} ({df_pred_2025.shape[0]:,} linhas x {df_pred_2025.shape[1]} colunas)")

# 3. Geração do previsoes.json no Contrato Oficial do Cartola FC
data_geracao_iso = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

lista_previsoes = []
for _, row in df_pred_2025.iterrows():
    lista_previsoes.append({
        "atleta_id": int(row["atleta_id"]),
        "ano": int(row["ano"]),
        "rodada_id": int(row["rodada_id"]),
        "clube_id": int(row["clube_id"]),
        "posicao_id": int(row["posicao_id"]),
        "pontos_predito": float(row["pontos_predito"]),
        "data_predicao": data_geracao_iso
    })

contrato_final_json = {"previsoes": lista_previsoes}

CAMINHO_PREVISOES_JSON = PROJECT_ROOT / "previsoes.json"
with open(CAMINHO_PREVISOES_JSON, "w", encoding="utf-8") as f:
    json.dump(contrato_final_json, f, indent=2, ensure_ascii=False)

print(f"\n=== ARQUIVO OFICIAL 'previsoes.json' GERADO COM SUCESSO! ===")
print(f"Caminho: {CAMINHO_PREVISOES_JSON} ({CAMINHO_PREVISOES_JSON.stat().st_size / (1024*1024):.2f} MB)")
print(f"Total de previsões geradas: {len(lista_previsoes):,} registros")

# Amostra do Contrato JSON
print("\nAmostra das 3 primeiras previsões no contrato oficial:")
print(json.dumps(lista_previsoes[:3], indent=2))